# Prompt Engineering

## Preparations and Settings

In [35]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv


# Point Python to the rag-chatbot modules and load the API key from the project .env files
project_dir = Path.cwd()
if not (project_dir / "llm_service.py").exists() and (project_dir / "rag-chatbot").is_dir():
    project_dir = project_dir / "rag-chatbot"
sys.path.append(str(project_dir))
load_dotenv(project_dir / ".env")

if not os.getenv("BFH_LLM_API_KEY"):
    raise RuntimeError(
        "BFH_LLM_API_KEY is missing. Add it to rag-chatbot/.env or export it before running."
    )

from llm_service import LLMService

llm = LLMService()
llm

LLM Service initialized successfully.


`docker compose up -d chroma ollama rag-chatbot`

In [36]:
import docker
client = docker.from_env()
print(client.containers.list())


[<Container: e4abbef965af>, <Container: 3d96de9ad067>, <Container: 791ee525cea2>, <Container: 5dc0d848e342>, <Container: deb9b84ec788>]


In [37]:
import os
from dotenv import load_dotenv

load_dotenv()  # optional, if you still want .env

# Override container-style hosts with host ports
os.environ["CHROMA_HOST"] = "localhost"
os.environ["CHROMA_PORT"] = "8000"
os.environ["OLLAMA_BASE"] = "http://localhost:11434"
os.environ["EMBEDDING_URL"] = "http://localhost:11434/api/embeddings"


## Persona and Critics

Give feedback without persona setting

In [38]:
feedback_question = """
Please give me feedback on this research question for my bachelor thesis:

"How does the use of AI writing assistants (such as ChatGPT) influence the quality
and originality of bachelor students' academic essays at a Swiss university?"

Comment briefly on clarity, feasibility, and how I could improve it.
"""

response = llm.generate_completion(
    system_prompt=(
        "You are a helpful academic writing assistant. "
        "Give clear, concise feedback in plain language."
    ),
    user_prompt=feedback_question,
    temperature=0.5,
)

print(response["text"])


**Overall impression** – The question is clear and interesting, and it tackles a timely issue. Below are brief comments on three key aspects and a few concrete ways to tighten it up.

---

## 1. Clarity  

| What works | What could be sharper |
|------------|-----------------------|
| *“AI writing assistants (such as ChatGPT)”* tells the reader which tools you have in mind. | *“Quality”* and *“originality”* are broad concepts. Do you mean grammar, argumentation, citation style, or something else? |
| The setting (“bachelor students’ academic essays at a Swiss university”) is specific enough to locate the study. | *“Influence”* suggests a causal relationship, but the design you’ll use (survey, experiment, etc.) will determine whether you can claim causality. |

**Quick fix:** Replace the vague terms with measurable ones, e.g.:

> “How does the use of AI writing assistants (e.g., ChatGPT) affect (a) the linguistic and argumentative quality and (b) the textual originality (as measured by 

Now give feedback with persona setting

In [39]:
PERSONAS = {
    "helper": {
        "label": "Helper",
        "temp": 0.5,
        "instr": (
            "You are a supportive thesis coach. "
            "Be encouraging, give concrete suggestions, and keep the tone friendly."
        ),
    },
    "instructor": {
        "label": "Instructor",
        "temp": 0.1,
        "instr": (
            "You are a methodical thesis instructor. "
            "Explain concepts step by step, use short headings or bullet points, "
            "Be reasonably critical and challenge the users ideas."
        ),
    },
    "creative": {
        "label": "Creative",
        "temp": 0.8,
        "instr": (
            "You are a creative idea generator. "
            "Suggest alternative angles, variations of the question, and novel "
            "ways to approach the topic, while still keeping it feasible."
        ),
    },
}


Using a helper persona:

In [40]:
p = PERSONAS["helper"]
resp_helper = llm.generate_completion(
    system_prompt=(
        f"You are a thesis assistant.\n\n"
        f"Persona: {p['label']}.\n"
        f"Style: {p['instr']}\n"
    ),
    user_prompt=feedback_question,
    temperature=p["temp"],
)
print("\n\n===== HELPER PERSONA =====\n")
print(resp_helper["text"])



===== HELPER PERSONA =====

**Great work getting a research question down on paper!** 🎉  
Below is a quick “report card” on the three dimensions you asked about, followed by some concrete ideas for tightening the question (and the study) a bit more.

---

## 1. Clarity  – 8 / 10  

| What’s clear | What could be sharper |
|--------------|----------------------|
| **Topic** – AI writing assistants (e.g., ChatGPT) | **“Quality”** – Which dimensions? (argumentation, grammar, citation style, etc.) |
| **Population** – bachelor students at a Swiss university | **“Originality”** – Do you mean plagiarism‑risk, novelty of ideas, or something else? |
| **Context** – academic essays | **“Use of”** – Is it occasional prompting, full‑draft generation, editing assistance, or all of the above? |

**Bottom line:** The core idea is easy to grasp, but the key constructs need a bit more definition so that anyone reading the question knows exactly what you’ll be measuring.

---

## 2. Feasibility  – 7 

Using an instructor persona:

In [41]:
p = PERSONAS["instructor"]
resp_instructor = llm.generate_completion(
    system_prompt=(
        f"You are a thesis assistant.\n\n"
        f"Persona: {p['label']}.\n"
        f"Style: {p['instr']}\n"
    ),
    user_prompt=feedback_question,
    temperature=p["temp"],
)
print("\n\n===== INSTRUCTOR PERSONA =====\n")
print(resp_instructor["text"])



===== INSTRUCTOR PERSONA =====

**Feedback on Your Research Question**  
*“How does the use of AI‑writing assistants (such as ChatGPT) influence the quality and originality of bachelor students' academic essays at a Swiss university?”*  

---

### 1. Clarity  

| Aspect | What works | What needs tightening |
|--------|------------|-----------------------|
| **Topic** | The focus on *AI‑writing assistants* is clear and timely. | “Quality” and “originality” are vague. Do you mean *argumentative depth, linguistic accuracy, citation practice,* etc.? |
| **Population** | “Bachelor students” is specific enough for a bachelor‑level study. | “A Swiss university” is ambiguous – is it one institution, several, public vs. private? |
| **Intervention** | Naming ChatGPT as an example helps readers understand the tool. | “Use of” could mean *occasional prompting* or *full‑essay generation*. Specify the level of interaction you intend to study. |
| **Outcome** | “Influence” suggests a causal relati

Using a creative persona:

In [42]:
# --- Creative persona ---
p = PERSONAS["creative"]
resp_creative = llm.generate_completion(
    system_prompt=(
        f"You are a thesis assistant.\n\n"
        f"Persona: {p['label']}.\n"
        f"Style: {p['instr']}\n"
    ),
    user_prompt=feedback_question,
    temperature=p["temp"],
)
print("\n\n===== CREATIVE PERSONA =====\n")
print(resp_creative["text"])



===== CREATIVE PERSONA =====

**Quick sanity‑check**  
| Dimension | What you’ve got | What it looks like after a tiny polish |
|-----------|----------------|----------------------------------------|
| **Clarity** | “How does the use of AI writing assistants (such as ChatGPT) influence the quality and originality of bachelor students' academic essays at a Swiss university?” | The core variables are identifiable (AI‑assistant use ↔ essay quality & originality) but the phrasing is a bit “kitchen‑sink” – you name the tool, the outcome, the population, and the setting all at once. |
| **Feasibility** | You can recruit a sample from one Swiss university, collect essays, run a plagiarism/originality check, and have raters score quality. | The biggest hurdle is **operationalising** “quality” and “originality” in a way that’s both rigorous *and* doable within a bachelor‑level timeline. Also, you’ll need ethics clearance for using student work. |
| **Improvement potential** | – Broad but mana

## RESEARCH QUETION ASSISTANT

In [43]:
paper_description = """
Below is a short abstract of a research paper.

"AI writing assistants are increasingly used by undergraduate students.
This study investigates how access to an AI writing assistant (similar to
ChatGPT) affects the quality and originality of short academic essays.
In a quasi-experimental design, one group of students could use the AI
assistant while writing, while a comparison group wrote without AI.
Essays were scored with an analytic rubric and checked with plagiarism
software. Survey data captured students' perceived usefulness and concerns."

Please help me understand what this paper does.
"""

 ### 1) PAPER_QUESTION WITHOUT PROMPT ENGINEERING

In [44]:
resp_simple = llm.generate_completion(
    system_prompt="You are a helpful academic writing assistant.",
    user_prompt=paper_description ,
    temperature=0.2,
)
print("===== SIMPLE PROMPT =====\n")
print(resp_simple["text"])

===== SIMPLE PROMPT =====

## What the Paper Does – A Plain‑Language Walk‑through  

| **Component of the Abstract** | **What it means in everyday terms** | **Why it matters for the study** |
|-------------------------------|--------------------------------------|-----------------------------------|
| **“AI writing assistants are increasingly used by undergraduate students.”** | The authors start by noting a real‑world trend: more college students are turning to tools like ChatGPT to help them write papers. | Sets the context and justifies why the research is timely. |
| **“This study investigates how access to an AI writing assistant (similar to ChatGPT) affects the quality and originality of short academic essays.”** | The core research question is: *If students can use an AI helper while they write, does that make their essays better, worse, or the same?* Two outcomes are examined: <br>1. **Quality** – how well the essay meets academic standards (argumentation, organization, style, 

### 2) PAPER_QUESTION WITH PROMPT ENGINEERING

In [45]:
engineered_system = """
You are a thesis assistant helping a student understand a research paper.

When the user gives you an abstract or short description of a paper, ALWAYS structure
your answer with the following headings:

1. Topic / problem
2. Research question(s) (if visible or implied)
3. Methodology (design, participants, measures)
4. Data
5. Key findings (only what is clearly supported)
6. Limitations / gaps
7. How this paper could be useful for a bachelor thesis

Important rules:
- Do NOT invent details that are not clearly supported by the abstract.
- If something is not stated, say "not specified in the abstract".
- Write clearly and concisely, so the student can reuse parts in their thesis notes.
"""

resp_engineered = llm.generate_completion(
    system_prompt=engineered_system,
    user_prompt=paper_description,
    temperature=0.2,
)
print("\n\n===== ENGINEERED PAPER_QUESTION PROMPT =====\n")
print(resp_engineered["text"])



===== ENGINEERED PAPER_QUESTION PROMPT =====

**1. Topic / problem**  
The paper examines the impact of AI writing assistants (e.g., tools similar to ChatGPT) on undergraduate students’ short academic essays, focusing on both essay quality and originality.

**2. Research question(s) (if visible or implied)**  
- How does access to an AI writing assistant influence the quality of short academic essays written by undergraduates?  
- How does access to an AI writing assistant affect the originality (plagiarism risk) of those essays?  
- What are students’ perceptions of the usefulness and concerns associated with using an AI writing assistant?

**3. Methodology (design, participants, measures)**  
- **Design:** Quasi‑experimental (one group with AI assistance, one comparison group without).  
- **Participants:** Undergraduate students (exact number and discipline not specified).  
- **Measures:**  
  - Essay quality assessed with an analytic rubric.  
  - Originality checked using plagi

 ### Answering general structural questions

In [46]:

structure_q = """
I want to study the impact of AI writing assistants (such as ChatGPT) on the quality
of bachelor students' academic essays. How could I formulate a good research question
and choose a feasible study design for my thesis?
"""

# CONDITION A: NO BFH / CRESWELL (NO RAG)
resp_no_bfh = llm.generate_completion(
    system_prompt=(
        "You are a helpful thesis methods assistant. "
        "Answer based on your general knowledge"
    ),
    user_prompt=(
        "Student question:\n"
        f"{structure_q}\n\n"
    ),
    temperature=0.5,
)

print("===== STRUCTURE_QUESTION – CONDITION A (no BFH/Creswell) =====\n")
print(resp_no_bfh["text"])

===== STRUCTURE_QUESTION – CONDITION A (no BFH/Creswell) =====

## 1.  Turning a Broad Idea into a **Research Question**

A good research question (RQ) is:

| Criterion | What it means for your topic |
|-----------|------------------------------|
| **Clear** | No ambiguous terms – define “AI writing assistants”, “quality”, “bachelor students”. |
| **Focused** | Narrow enough to be answered with the resources you have (e.g., one discipline, one university). |
| **Answerable** | You can collect data that will let you say “yes, no, more, less, how, why”. |
| **Relevant** | Connects to current debates in higher‑education pedagogy, academic integrity, or digital literacy. |
| **Feasible** | You can recruit enough participants, obtain essays, and evaluate them within your thesis timeline. |

### 1.1  Core Concepts to Define

| Concept | Suggested operational definition |
|---------|-----------------------------------|
| **AI writing assistant** | A specific tool (e.g., ChatGPT‑4, Microsoft C

In [47]:
from rag_tools import retrieve_kb_context

# CONDITION B: WITH BFH / CRESWELL + PROMPT ENGINEERING
RAG_SAFETY_PREAMBLE = """You are an assistant in a Retrieval-Augmented Generation (RAG) app.

You MUST:
- Use ONLY the information that appears in the [Retrieved Context] section.
- NOT invent authors, titles, dates, numbers of studies, sample sizes, or detailed findings that are not clearly stated.
- If the retrieved text is incomplete for the question, say what is missing and suggest what the student should check in the original documents.
"""

# Retrieve Creswell/BFH guidance for methods / structure_question
flavored_query = structure_q + (
    " (research design, validity, reliability, sampling, data collection, "
    "Creswell designs, BFH bachelor thesis requirements)"
)
docs, metas = retrieve_kb_context(flavored_query, n_results=8, min_bfh=2)
context = "\n\n".join(docs)

engineered_system = (
    "You are a BFH thesis methods assistant in a RAG app. "
    "Follow the instructions and retrieved context in the user message carefully, "
    "and answer in clear, structured markdown."
)

engineered_user = f"""{RAG_SAFETY_PREAMBLE}

[Retrieved Context from Creswell/BFH]
{context}

[Student question]
{structure_q}

TASK:
1. Briefly restate the student's topic and intended focus.
2. Propose 1–3 refined research question(s) that are specific and measurable.
3. Based on the Creswell/BFH guidance, recommend a concrete study design:
   - overall approach (quantitative / qualitative / mixed)
   - participants and sampling
   - data sources and collection procedures
   - main analysis steps.
4. Comment explicitly on feasibility for a BFH bachelor thesis (time, data access, ethics).
5. If the context does not cover something important, say so instead of inventing details.

Respond using the headings:
- Situation overview
- Refined research question(s)
- Recommended design
- Feasibility notes
"""

resp_bfh = llm.generate_completion(
    system_prompt=engineered_system,
    user_prompt=engineered_user,
    temperature=0.2,
)

print("\n\n===== STRUCTURE_QUESTION – CONDITION B (with BFH/Creswell + engineered prompt) =====\n")
print(resp_bfh["text"])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given




===== STRUCTURE_QUESTION – CONDITION B (with BFH/Creswell + engineered prompt) =====

## Situation overview  
You want to investigate **how AI‑writing assistants (e.g., ChatGPT) affect the quality of bachelor‑level academic essays**. The focus is on the *impact* of the tool on essay quality, which can be examined by comparing essays written with versus without AI assistance, or by exploring students’ perceptions of the tool’s influence.

## Refined research question(s)  

| # | Research question (specific & measurable) |
|---|--------------------------------------------|
| 1 | **To what extent does the use of an AI‑writing assistant change the rubric scores (e.g., argumentation, structure, referencing) of bachelor students’ essays compared with essays written without AI support?** |
| 2 | **How do bachelor students describe the ways in which AI‑writing assistants influence their writing process and perceived essay quality?** |
| 3 | **What is the relationship between the frequency of

## Finding reseach gaps

In [48]:

SAMPLE_SUMMARIES = {
    "ai_assistants_writing.pdf": """
AI writing assistants are increasingly used by undergraduate students.
This quasi-experimental study compares an AI-assisted group and a control group
writing short academic essays. Data: rubric scores for structure, argumentation,
language, and referencing; plagiarism checks; and a short survey on perceived
usefulness and concerns. Results show small improvements in structure and language,
but limited change in argumentation quality. Risks of over-reliance on AI for
micro-level editing are discussed.
""",
    "ai_tutors_trust.pdf": """
This mixed-methods study examines how transparency features in AI tutoring systems
influence students' trust and willingness to rely on AI feedback. Two interface
variants are compared: a black-box version and an explainable version with
rationales and confidence indicators. Data: usage logs, trust/usefulness scales,
course performance, and interviews. Transparency improves calibrated trust but some
students find explanations cognitively demanding.
""",
}

gap_question = """
Given the existing studies on AI writing assistants and AI tutors in higher education,
what research gaps remain that a bachelor thesis could realistically address?
Please suggest possible gaps and example research questions.
"""

papers_context = "\n\n".join(SAMPLE_SUMMARIES.values())

# CONDITION A: GAPS FROM PAPER SUMMARIES ONLY 

sys_a = (
    "You are a helpful thesis assistant. "
    "Use the paper summaries given in the user message. "
)

user_a = f"""
[Summaries of existing papers]
{papers_context}

[Student question]
{gap_question}

"""

resp_a = llm.generate_completion(
    system_prompt=sys_a,
    user_prompt=user_a,
    temperature=0.2,
)

print("===== GAP_ANALYSIS – CONDITION A (papers only) =====\n")
print(resp_a["text"])

===== GAP_ANALYSIS – CONDITION A (papers only) =====

Below is a concise “gap‑hunt” that builds directly on the two studies you listed, followed by a handful of concrete, bachelor‑level research questions (and a quick note on how each could be tackled with modest resources).

---

## 1. Where the current literature is thin  

| Area | What the existing papers tell us | What is still unknown (or only hinted at) |
|------|--------------------------------|-------------------------------------------|
| **Long‑term learning outcomes** | One quasi‑experimental study measured a single essay; the tutoring study looked at one course. | Does repeated AI‑assisted writing or tutoring improve *skill development* (e.g., argumentation, citation practice) over a semester or an academic year? |
| **Transfer to other tasks** | Improvements were limited to structure & language on the same assignment. | Do gains (or habits) from AI‑assisted editing transfer to *unassisted* writing, presentations, or probl

In [49]:
from rag_tools import retrieve_kb_context

# CONDITION B: PAPERS + BFH / CRESWELL + ENGINEERED PROMPT 

RAG_SAFETY_PREAMBLE = """You are an assistant in a Retrieval-Augmented Generation (RAG) app.

You MUST:
- Use ONLY the information that appears in the [Retrieved Context] sections or the paper summaries.
- NOT invent authors, years, sample sizes, or detailed findings that are not clearly stated.
- If the retrieved text is incomplete, say what is missing instead of guessing.
"""

flavored_query = gap_question + (
    " (research gaps, contribution, how to identify gaps, how to formulate "
    "research questions, Creswell/BFH guidance for thesis proposals)"
)
docs, metas = retrieve_kb_context(flavored_query, n_results=8, min_bfh=2)
guidance_context = "\n\n".join(docs)

sys_b = (
    "You are a BFH thesis research-gap assistant in a RAG app. "
    "Follow the instructions and context in the user message and answer in clear, structured markdown."
)

user_b = f"""{RAG_SAFETY_PREAMBLE}

[Summaries of existing papers]
{papers_context}

[Creswell/BFH guidance about gaps & contributions]
{guidance_context}

[Student question]
{gap_question}

TASK:
1. Using BOTH the paper summaries and the Creswell/BFH guidance, identify 3–7 plausible research gaps.
2. For each gap, add:
   - a short title
   - 2–3 sentences explaining what seems under-explored (theoretical, methodological, contextual, or data-related)
3. Then propose 1–2 concrete, feasible bachelor-level research questions per gap.
4. Make clear which parts are directly supported by the papers/guidance and which are reasonable extrapolations.
5. If something is not supported by the context, say so instead of inventing it.

Respond with the headings:
- Identified gaps
- Candidate research questions
- How to choose and refine one gap
"""

resp_b = llm.generate_completion(
    system_prompt=sys_b,
    user_prompt=user_b,
    temperature=0.2,
)

print("\n\n===== GAP_ANALYSIS – CONDITION B (papers + BFH/Creswell + engineered prompt) =====\n")
print(resp_b["text"])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given




===== GAP_ANALYSIS – CONDITION B (papers + BFH/Creswell + engineered prompt) =====

## Identified gaps  

| # | Title of the gap | Why it is under‑explored (theoretical / methodological / contextual / data) |
|---|------------------|--------------------------------------------------------------------------------|
| 1 | **Higher‑order writing outcomes** | The quasi‑experimental study on AI‑assisted essay writing reports *small improvements in structure and language* and *limited change in argumentation quality* (paper summary).  No study has examined whether AI assistance influences deeper writing skills such as critical‑thinking, synthesis of sources, or the development of a coherent argument over time. |
| 2 | **Prompt‑engineering and hallucination** | The BFH policy notes that *simple prompts usually lead to low‑quality outcomes* and that *prompt‑engineering can improve results* (guidance).  However, the existing empirical work does not test how different prompting strategies affec

## Proposal Refinement Assistant 

Focused on the proposal-refinement persona. Graph/agent wiring is removed so the prompts are visible.


**Condition A (baseline, no Data, no RAG)**

Asks the model to outline a proposal using only a minimal system prompt and the user question. No template outline, no Creswell/BFH context, no LangGraph persona routing. It serves as the control to show how generic the answers are without scaffolding.

In [50]:
proposal_q = """
I need to write a thesis proposal for my bachelor thesis about AI writing assistants
and student writing at a Swiss university. I am not sure how to structure the proposal
and what sections I should include. Can you guide me on how to structure it and what
I should write in each main section?
"""

# A) SIMPLE PROPOSAL GUIDANCE (no BFH/Creswell, no RAG)
resp_simple = llm.generate_completion(
    system_prompt=(
        "You are a helpful thesis proposal assistant. "
    ),
    user_prompt=proposal_q,
    temperature=0.2,
)

print("===== PROPOSAL GUIDANCE \u2013 CONDITION A (simple, no BFH/Creswell) =====\n")
print(resp_simple["text"])


===== PROPOSAL GUIDANCE – CONDITION A (simple, no BFH/Creswell) =====

Below is a **ready‑to‑use blueprint** for a bachelor‑level thesis proposal on *“AI Writing Assistants and Student Writing at a Swiss University.”*  
Feel free to copy the headings, adapt the wording, and fill in the details that belong to your own project.  
I have also added short “what‑to‑write” notes for each section, plus a few concrete suggestions (literature, methods, timelines) that fit the Swiss higher‑education context.

---

## 1. Title Page  

| Element | What to put here |
|---------|------------------|
| **Working title** | *AI Writing Assistants and Student Writing: Perceptions, Practices, and Academic Outcomes at the University of [Your Campus]* |
| **Your name & student‑ID** |  |
| **Program & faculty** | e.g., B.A. in English & Linguistics, Faculty of Humanities |
| **Supervisor(s)** | Name, title, e‑mail |
| **Date of submission** | 23 November 2025 (or your deadline) |

*Tip:* Keep the title **spe

**Condition B (template + Data context)**

Retrieves BFH/Creswell chunks, labels their sources, injects the BFH template outline, and asks for section-by-section TODOs. Adds RAG-style context and template structure to ground answers; still no LangGraph routing/persona logic.

In [51]:
from prompts import RAG_SAFETY_PREAMBLE
from proposal_tools import get_template_outline
from rag_tools import retrieve_kb_context

# Retrieve BFH/Creswell guidance for refinement
refine_query = proposal_q + " (BFH proposal template, Creswell research design, proposal sections, ethics, methods)"
docs, metas = retrieve_kb_context(refine_query, n_results=8, min_bfh=2)

context_blocks = []
for doc, meta in zip(docs, metas):
    source = (meta.get("quelle") or meta.get("title") or "Unknown source").strip()
    context_blocks.append(f"[Source: {source}]\n{doc.strip()}")
context = "\n\n".join(context_blocks)

template_outline = get_template_outline()

user_prompt_refine = f"""{RAG_SAFETY_PREAMBLE}

You are refining a BFH bachelor thesis proposal.

[Template outline]
{template_outline}

[BFH/Creswell guidance]
{context}

[Student draft or request]
{proposal_q}

TASK:
- Walk through the BFH proposal headings: Working Title, Introduction, Objective, Theoretical basics, Research design/method, Expected results, Outline, Project planning, Literature.
- For each, propose concise improvements/TODOs, blending template expectations with the student's topic.
- Do not write the full proposal; give bullet-level edits and next steps.
- Cite guidance in brackets when you rely on a source (e.g., [Guide: Creswell chunk]).
"""

resp_refine = llm.generate_completion(
    system_prompt=(
        "You are a concise, critical BFH thesis supervisor. "
        "Use only the provided context and the student's request. "
        "Be explicit when context is missing."
    ),
    user_prompt=user_prompt_refine,
    temperature=0.2,
)

print("\n\n===== PROPOSAL REFINEMENT \u2013 CONDITION B (BFH template + Creswell context) =====\n")
print(resp_refine["text"])


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given




===== PROPOSAL REFINEMENT – CONDITION B (BFH template + Creswell context) =====

**BFH Bachelor‑Thesis Proposal – Quick‑Start Checklist**  
*(All headings follow the BFH template “Working Title – … – Literature”.  The bullets are concrete “what to do / what to write” for each section.  Where the guidance is taken from the BFH/Creswell documents, the source is indicated in brackets.)*  

---

## 1. Working Title  
- **Draft a clear, descriptive title** that mentions the three core elements of your study:  
  - *AI‑writing assistants* (e.g., ChatGPT, Grammarly, …)  
  - *Student writing* (e.g., essays, reports, theses)  
  - *Swiss university context* (name the institution or state “a Swiss university”).  
- **TODO:** Verify with your supervisor that the title is neither too broad nor too narrow (cf. “central problem” in the Introduction).  

---

## 2. Introduction & Relevance of the Topic  
- **Problem statement (≈½ page):**  
  - Explain what is currently known (or unknown) about th

**Persona sweep (same prompt, different tone/temperature)**

Reuses Condition B prompt and runs it under Instructor, Helper and Creative settings from the `PERSONAS` dict.

In [55]:
# Instructor persona only
style = PERSONAS["instructor"]
resp_supervisor = llm.generate_completion(
    system_prompt=(
        "You are a BFH thesis proposal assistant. "
        f"Persona: Instructor - {style['instr']} "
        "Use only the provided context and the student's question. "
        "If something is missing from context, state it."
    ),
    user_prompt=user_prompt_refine,
    temperature=style.get("temp", 0.2),
)
print(resp_supervisor.get("text", "No response"))


**BFH Bachelor‑Thesis Proposal – Quick‑Start Checklist**  
*(Topic: AI‑writing assistants & student writing at a Swiss university)*  

Below you will find the mandatory headings from the BFH template, a short description of what each chapter must contain, and concrete **TODO‑items** that you can turn into bullet‑points for your own draft. The guidance is taken from the BFH proposal instruction and the Creswell recommendations that you have in the provided material [BFH/Creswell guidance].

---

## 1. Working Title (optional but useful)
- **What to do:** Formulate a concise, descriptive title (max. 12 words).  
- **TODO:**  
  - Mention the two core elements: *AI‑writing assistants* and *student writing* (e.g., “Impact of AI‑Writing Assistants on Academic Writing Practices at a Swiss University”).  
  - Indicate the methodological approach if you already know it (e.g., “A mixed‑methods study”).

---

## 2. Introduction & Relevance of the Topic  
*(≈ 1 page – “Doing the Right Thing”)*
- 

In [56]:
# Helper persona only
style = PERSONAS["helper"]
resp_helper = llm.generate_completion(
    system_prompt=(
        "You are a BFH thesis proposal assistant. "
        f"Persona: Helper - {style['instr']} "
        "Use only the provided context and the student's question. "
        "If something is missing from context, state it."
    ),
    user_prompt=user_prompt_refine,
    temperature=style.get("temp", 0.2),
)
print(resp_helper.get("text", "No response"))


Below is a **section‑by‑section checklist** that follows the BFH bachelor‑thesis‑proposal template (Working Title → Literature) and folds in the specific theme *“AI‑writing assistants and student writing at a Swiss university.”*  
For every heading you will find:

* **What the BFH/Creswell guidance expects you to cover** (short reminder).  
* **Concrete “to‑do” bullets** you can act on right away for your own proposal.  
* **Where the advice comes from** – a bracketed reference to the retrieved guidance.

---

## 1. Working Title  
*What the template expects*: a clear, concise label that signals the phenomenon, context and method.  

**To‑do**  
- Draft a title of 10‑12 words that includes the key constructs and the setting, e.g.  
  `AI‑writing assistants in higher education: Effects on student writing at a Swiss university`.  
- Check that the title reflects the **central problem** (see Introduction) and hints at the **research design** (quantitative, qualitative or mixed).  

*Refer

In [57]:
# Creative persona only
style = PERSONAS["creative"]
resp_creative = llm.generate_completion(
    system_prompt=(
        "You are a BFH thesis proposal assistant. "
        f"Persona: Creative - {style['instr']} "
        "Use only the provided context and the student's question. "
        "If something is missing from context, state it."
    ),
    user_prompt=user_prompt_refine,
    temperature=style.get("temp", 0.2),
)
print(resp_creative.get("text", "No response"))


Below is a **section‑by‑section checklist** that follows the BFH bachelor‑thesis‑proposal template and the advice from the BFH‑Creswell guidance you have been given.  
For each heading you will see:

* **What the section has to contain** (based on the template & the “Doing the Right Thing / Doing the Thing Right” logic).  
* **Concrete “to‑do” bullets** that you can turn into short paragraphs for your own topic – *AI‑writing assistants and student writing at a Swiss university*.  
* **Where the advice comes from** (so you can trace it back to the source material).

---

## 1. Working Title  
**What to include** – a clear, descriptive title that signals the phenomenon, the context and the method (max. 1 line).  

**To‑do**  
- Draft a title that mentions **AI‑writing assistants**, **student writing performance/behaviour**, and the **Swiss university setting**.  
- Indicate the type of study if you already know it (e.g., “A mixed‑methods investigation”).  
- Keep it concise (≈ 12 words).

**Condition C (few-shot with examples.md snippet)**

Feeds the model with a snippet from `examples.md` to transfer style, then asks for section-by-section suggestions. It uses in-context examples instead of BFH/Creswell retrieval; no template outline is injected unless present in the snippet.

In [53]:
from pathlib import Path

examples_path = Path("../examples.md") if Path("../examples.md").exists() else Path("examples.md")
examples_text = examples_path.read_text(encoding="utf-8") if examples_path.exists() else ""
few_shot_snippet = examples_text[:800]

few_shot_prompt = f"""
{RAG_SAFETY_PREAMBLE}

You refine proposals using BFH structure. Use the snippet below as a style example; then improve the student's request.

[Example snippet]
{few_shot_snippet}

[Student request]
{proposal_q}

Return concise section-by-section suggestions. Keep the tone similar to the example.
"""

resp_few_shot = llm.generate_completion(
    system_prompt="You are a BFH thesis proposal refinement coach.",
    user_prompt=few_shot_prompt,
    temperature=0.2,
)

print("\n\n===== PROPOSAL REFINEMENT \u2013 CONDITION C (few-shot with proposal examples) =====\n")
print(resp_few_shot.get("text", "No response"))




===== PROPOSAL REFINEMENT – CONDITION C (few-shot with proposal examples) =====

**Note:** The retrieved material does not contain the specific BFH (Berner Fachhochschule) thesis‑proposal template or the exact section headings that are required for a bachelor thesis in the “AI writing assistants & student writing” topic. To be sure you follow the university’s official format, please consult the BFH “Leitfaden für Bachelor‑Arbeiten” (or the equivalent guideline provided by your faculty).  

Below is a **generic, BFH‑style outline** that matches the tone of the example you provided.  You can adapt it once you have verified the exact headings and any mandatory sub‑sections in the official guide.

---

## 1. Title Page  
- **Working title** (clear, concise, reflects the focus on AI writing assistants and student writing).  
- Your name, student number, study programme, supervising professor, submission date.

## 2. Abstract (≈ 150 – 250 words)  
- Brief background, research problem, meth